# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working rule: keep the decision in front of the model, not the model in front of the decision.

We are working in the FlyRank refresh / content-opportunity lane: which pages deserve a human review first?

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a **ranking / scoring** task. The decision is not “is this page good?” in an absolute sense; it is “which pages should a human editor review first?” That means we want a priority score and a ranked queue. A higher score should correspond to higher review value.

In [ ]:
lane_type = "ranking / scoring"
decision = "Which pages should a human editor review first?"

print(f"Lane type: {lane_type}")
print(f"Decision: {decision}")


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target is a page-level signal of likely decline or review-worthy weakness over a later window. In the starter repo, a simple, observable proxy is `trend_direction == "down"` for content with enough evidence. That is an observed pattern from data, not a product rule. I would not use FlyRank product flags as features, because those are already outputs of the company decision system.

In [ ]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
]

data_path = next((p for p in candidate_paths if p.exists()), candidate_paths[0])
df = pd.read_csv(data_path)

df["is_declining_proxy"] = (df["trend_direction"] == "down").astype(int)

print("Rows:", len(df))
print(df[["content_id", "trend_direction", "is_declining_proxy"]].head().to_string(index=False))
print("Proxy rate:", round(df["is_declining_proxy"].mean(), 3))


## 3. Success metric

*One metric you can defend. What number means 'good'?*

For a prioritized review queue, the defendable metric is **Precision@K**. We choose K based on reviewer capacity; for example, if a reviewer can inspect 50 pages, then **Precision@50** is the metric. A good model is one that places many genuinely review-worthy pages in the top K, rather than just being broadly accurate.

In [ ]:
review_capacity = 50
metric = f"Precision@{review_capacity}"

print(f"Chosen metric: {metric}")
print("Interpretation: among the top 50 pages, how many are genuinely review-worthy or declining?")


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row is one **pseudonymized content item**. The dataframe is a content-level slice with observed search and engagement signals from the prior period. I will use a screening rule like enough impressions and enough age to keep the signal meaningful, but no client-identifying fields.

In [ ]:
analysis_df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
analysis_df = analysis_df[[
    "content_id", "client_id", "content_type", "main_intent",
    "impressions_90d", "ctr", "avg_position", "trend_direction",
    "word_count", "content_age_days"
]].copy()

print("Rows after screening:", len(analysis_df))
print(analysis_df.head(5).to_string(index=False))


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The signal is too tangled for a simple hand-written rule because a page can look weak for multiple reasons at once: low CTR, weak visibility, stale content, bad position, and varying demand by content type. A single if/then rule cannot capture the combined effect of these features without becoming brittle or overly simplistic. A model can learn interactions across search volume, click-through rate, position, freshness, and content type.

In [ ]:
summary = df.groupby("content_type")[["ctr", "avg_position", "impressions_90d"]].mean().round(3)

print("Grouped signal summary:")
print(summary.to_string())
print("
Observed pattern: different content types have different search and engagement profiles, so a single static rule is unlikely to be robust.")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom
- [ ] The task is framed as a decision, not a vague prediction goal
- [ ] The target is observable or a clearly labeled proxy, not a product rule disguised as truth
- [ ] The metric matches the actual review decision
- [ ] The unit of analysis is a real row in a real dataframe
- [ ] The claim about ML beating a fixed rule is grounded in messy real signals, not magic
- [ ] The language stays observed / directional / decision-support

If a box is still blank, do the missing part before you submit.

In [ ]:
checks = {
    "task_is_a_decision": True,
    "target_is_observed_or_proxy": True,
    "metric_matches_review_capacity": True,
    "row_is_one_content_item": True,
    "ml_beats_rule_for_messy_interactions": True,
    "language_is_public_safe": True,
}

for key, value in checks.items():
    print(f"{key}: {value}")
